# MAKE SURE TO CLEAR ALL CELLS BEFORE PUSHING

In [ ]:
from nqct.utils.QuantumSession import QuantumSession

qs = QuantumSession(api_key=input(), storage_path='temp/')
qs.list_backends();
qs.select_backend('quantware_soprano_d2')

In [ ]:
my_script = r"""
OPENQASM 3;
include 'stdgates_transmon_fixed_coupler.inc';
 
bit[2] c;
qubit[2] q;
 
z q[0];
rx(pi/2) q[0];
 
z q[1];
ry(pi/2) q[1];
 
ctrl @ z q[0], q[1];
 
z q[1];
ry(pi/2) q[1];
 
delay[0]  q[0], q[1];
c[0] = measure q[0];
c[1] = measure q[1];
"""
qs.set_qasm(my_script)
qs.get_qregs_in_qasm()
qs.set_qreg_physical_mapping({('q',0):1, ('q',1):2})
qs.validate()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
t_vals = np.arange(0,60e-9,1/2.4e9)
f_vals = 1-np.exp(-t_vals/10e-9)*np.cos(2*np.pi/30e-9*t_vals)
f_vals[-1] = 0
plt.plot(f_vals)

In [ ]:
import numpy as np
# qs.declare_numpy_waveform("wfm_envelope", np.array([1+2j,1-1j,0,3+4j,2j]))


my_script = r"""
OPENQASM 3;
include 'stdgates_transmon_fixed_coupler.inc';

bit[2] c;
qubit[2] q;

defcal cz $0, $1
{
play(flux($2), wfm_flux);
}

z q[0];
rx(pi/2) q[0];

z q[1];
ry(pi/2) q[1];

cz q[0], q[1];

z q[1];
ry(pi/2) q[1];

delay[0]  q[0], q[1];
c[0] = measure q[0];
c[1] = measure q[1];
"""
qs.set_qasm(my_script)

qs.declare_numpy_waveform("wfm_flux", f_vals)
print(qs.get_final_qasm())

In [ ]:
qs.validate()

In [ ]:
qs.set_acquisition_type('DISCRIMINATION')
qs.set_averaging_type('SingleShotCounts')
qs.set_num_shots(1024)
qs.set_shot_repeat(1)
leRes = qs.run()
leRes.get_inner_slicing_vars()

In [ ]:
import numpy as np
cur_arrs = leRes.get_data('c')

combinations = np.array([[x,y] for x in [0,1,2] for y in [0,1,2]])
counts = []
for cur_comb in combinations:
    counts.append( int(np.sum((cur_arrs[0] == cur_comb[0]) & (cur_arrs[1] == cur_comb[1]))) )
counts = np.array(counts)
counts, combinations

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1)

ax.bar([str(x) for x in combinations], counts)

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

ax.bar3d(combinations[:,0], combinations[:,1], counts*0, 1,1, counts)
ax.set_xticks([0.5,1.5,2.5])
ax.set_xticklabels(['0','1','2'])
ax.set_yticks([0.5,1.5,2.5])
ax.set_yticklabels(['0','1','2'])